# Spotify: estimar la popularidad de una canción
## Unidad 2 · Objetivo, separación y referencia de validación

Este cuaderno utiliza la base preparada en la unidad 1. Incluye la definición del problema de regresión, la separación entre entrenamiento y prueba interna y una validación agrupada de la referencia de mediana dentro de entrenamiento. Cada decisión se explica antes de ejecutar el código.

**Archivos en la misma carpeta:**

```text
Spotify_Unidad_2.ipynb
Spotify_Base_Unidad_2.csv
```

La base preparada se adjunta para ejecutar este cuaderno de forma independiente. No es el CSV original renombrado: procede de `df_base_canciones` de la unidad 1 y añade una clave auxiliar para agrupar versiones identificadas de una canción.

**Alcance actual:** evaluamos la referencia de mediana con tres pliegues dentro de entrenamiento. No ajustamos todavía modelos con características, imputadores o escaladores. La prueba interna permanece reservada.

## 1. Objetivo de predicción

**Objetivo general:** entrenar y comparar modelos que estimen el puntaje de popularidad registrado de una canción a partir de sus características y géneros, y comprobar si mejoran una referencia sencilla en una evaluación interna documentada.

**Qué predice:** un número en la escala de popularidad 0–100. Por eso comenzamos con **regresión**.

**Información de entrada:** diez medidas musicales, tres categorías y 114 indicadores de género. Popularidad es la respuesta que el modelo debe aprender a estimar y no forma parte de sus entradas.

**Uso inicial:** estudiar si esas variables permiten aproximar diferencias de popularidad dentro de esta muestra. No afirmamos todavía que el modelo anticipe el éxito de un lanzamiento futuro. Eso exigiría fechas y datos disponibles antes del momento de predicción.

**Decisión de simplicidad:** empezamos por una tarea central. Clasificación y agrupamiento responden preguntas distintas y solo se incorporarán si aportan al objetivo o a los requisitos de la asignatura.

### 1.1 Qué conservamos de la unidad 1

La base tiene **89.740 IDs**, una observación por ID, con las 114 pertenencias de género. Las **720 discrepancias de popularidad por ID** se resumieron mediante mediana y mantienen su limitación de significado temporal.

Tempo y compás conservan ausencias. Compás 1 y sonoridad positiva conservan sus advertencias. No eliminaremos esas observaciones automáticamente para simplificar el modelamiento.

La base incluye duración en milisegundos y minutos. Usaremos **solo minutos** en las entradas para evitar dos columnas equivalentes.

Los datos están en sus unidades originales. **No reutilizamos `df_demo_estandarizado` ni `df_demo_minmax`**, ajustados con toda la muestra para aprender las fórmulas. Cuando corresponda, imputación y escalamiento se ajustarán solo con entrenamiento.

## 2. Qué mediremos: MAE y referencia

**Métrica principal: MAE**, promedio de las diferencias absolutas entre popularidad real y estimada. Se expresa en puntos; menor es mejor.

**Ejemplo ficticio:** errores de 5, 10 y 15 puntos producen un MAE de 10. No equivale a 90% de aciertos ni limita todos los errores a diez puntos.

**Referencia propuesta:** predecir siempre la mediana de popularidad del entrenamiento. Es una comparación sencilla y coherente con el error absoluto; se calculará después de separar los datos y se aplicará sin aprender de prueba.

**Mejora relativa de MAE:**

`(MAE de referencia − MAE del modelo) / MAE de referencia × 100`

Si la referencia tuviera MAE 20 y el modelo 18, la mejora sería 10%. Es un ejemplo, no un resultado de este proyecto. Si la referencia tuviera error cero, no se calcularía esa división: ya sería perfecta según esta métrica.

**Complementos:** RMSE informa sobre errores grandes; R² compara error cuadrático con una referencia de la media y puede ser negativo. Los reportaremos más adelante sin convertirlos en porcentajes de acierto.

### 2.1 KPI fijados antes de modelar

| Indicador | Meta de trabajo | Por qué |
|---|---|---|
| IDs compartidos entre entrenamiento y prueba | 0 | Una canción no debe estar en ambos conjuntos |
| Claves artista + título compartidas | 0 | Mantener juntas las versiones reconocidas de una canción |
| Cobertura de la partición | 100% de los IDs preparados, una sola vez | No perder ni duplicar observaciones al separar |
| Mejora relativa del MAE frente a la referencia | Al menos 10% | Exigir una mejora cuantificable sobre una estrategia sencilla |

**El 10% es una meta inicial propuesta para el ejercicio**, no un umbral universal, requisito confirmado de una rúbrica o promesa de resultado. Se fija ahora para no cambiarlo después de ver el desempeño. No demuestra suficiencia para decisiones comerciales; aún falta una tolerancia de error definida por el uso.

Los tres primeros indicadores se calculan en este paso. El de predicción queda pendiente hasta entrenar y evaluar. La selección de modelos y configuraciones se hará dentro de entrenamiento; prueba no se usará para buscar una combinación ganadora.

RMSE, R², tiempos y errores por grupos serán evidencia complementaria, sin añadir por ahora más metas arbitrarias.

## 3. Qué significa nuestra prueba interna

**Entrenamiento:** datos disponibles para aprender parámetros y decidir preparación y modelos mediante validación interna.

**Prueba:** conjunto apartado para el informe final del procedimiento seleccionado. No se usa para escoger variables, ajustar umbrales, imputar o escalar.

**Limitación de este proyecto:** el dataset completo ya se describió en la unidad 1 y sus decisiones de preparación se tomaron con esa información. Separarlo ahora no borra ese conocimiento. Por tanto, esta será una **evaluación interna con exposición exploratoria previa**, no una prueba externa completamente ciega.

Para evidencia más independiente se necesitaría una muestra nueva o externa no inspeccionada, y para afirmar predicción futura, una separación temporal apropiada. Esta limitación se conservará en las conclusiones.

### 3.1 Por qué agruparemos versiones de canciones

IDs diferentes pueden corresponder a reediciones de una canción con artista y título iguales. Si quedan en ambos conjuntos, el problema puede resultar artificialmente fácil.

La base incorpora `clave_cancion`, formada por **artistas + título normalizados**: minúsculas, espacios externos retirados y espacios repetidos reducidos. Antes de exportar se comprobó que los textos no variaran dentro de cada ID.

**Ejemplo ficticio:** «Artista A || Canción X» con dos IDs permanece completamente en entrenamiento o completamente en prueba.

**Alcance:** esto no separa artistas completos. Un artista puede tener títulos distintos en ambos conjuntos. Tampoco detecta todas las versiones con títulos diferentes, distintos créditos o distinto orden de colaboradores; no es deduplicación musical perfecta.

La clave sirve únicamente para la partición y no se entrega al modelo como predictora.

## 4. Cargar la base preparada

**Línea 1:** importa Pandas.

**Línea 2:** guarda el nombre exacto del CSV preparado.

**Línea 3:** `pd.read_csv(...)` carga ese archivo local. `dtype` indica texto para ID y clave y entero nullable `Int64` para compás, para preservar sus ausencias sin convertir códigos en decimales por necesidad de representación.

**Línea 4:** `.set_index("track_id")` coloca el identificador como índice. No es una columna de entrada musical.

**Línea 5:** muestra las dimensiones después de esa operación.

In [1]:
import pandas as pd
archivo_base = "Spotify_Base_Unidad_2.csv"
base_unidad2 = pd.read_csv(archivo_base, dtype={"track_id": "string", "clave_cancion": "string", "time_signature": "Int64"})
base_unidad2 = base_unidad2.set_index("track_id")
base_unidad2.shape

(89740, 130)

**Resultado:** **89.740 filas y 130 columnas**, sin contar el índice. Son las 129 columnas de la base integrada y una clave de partición. El CSV incluye también `track_id` como campo para reconstruir el índice.

No todas serán entradas: se excluyen popularidad, la clave y la duración redundante. El CSV original y la base de unidad 1 permanecen intactos.

### 4.1 Comprobar los requisitos de la separación

**Línea 1:** crea una Serie de verificaciones: índice único, ID presente, clave presente y clave no vacía después de quitar espacios.

**Línea 2:** muestra sus resultados.

En la tercera celda, `if not ...all()` detiene el proceso si algún requisito falla. Así no asignamos a ciegas observaciones con ID o clave ausentes. `raise ValueError(...)` solicita revisar el archivo, no elimina registros.

In [2]:
requisitos_particion = pd.Series({
    "ID único": base_unidad2.index.is_unique,
    "ID sin faltantes": base_unidad2.index.notna().all(),
    "Clave sin faltantes": base_unidad2["clave_cancion"].notna().all(),
    "Clave sin texto vacío": base_unidad2["clave_cancion"].str.strip().ne("").all()
})
requisitos_particion

ID único                 True
ID sin faltantes         True
Clave sin faltantes      True
Clave sin texto vacío    True
dtype: bool

In [3]:
if not requisitos_particion.all():
    raise ValueError("Revisar IDs y claves antes de realizar la partición.")

## 5. Separar aproximadamente 80/20 por grupos

**Decisión:** destinar el 80% de las **claves distintas** a entrenamiento y el resto a prueba. Las claves pueden reunir cantidades distintas de IDs; por eso los porcentajes de filas serán aproximados, no obligatoriamente 80,00/20,00.

Usamos Pandas para que se vea la mecánica de separar grupos. Es una partición agrupada, no un `train_test_split` aleatorio por filas. No necesita ajustar ningún modelo.

**Línea 1:** `.drop_duplicates()` obtiene claves únicas, `.sort_values()` fija su orden y `.reset_index(drop=True)` crea posiciones consecutivas.

**Línea 2:** `.sample(frac=0.80, random_state=42)` toma aproximadamente 80% de esas claves, sin reemplazo por defecto. 42 fija una semilla; no mejora el resultado ni es una métrica.

**Línea 3:** `.isin(claves_entrenamiento)` crea la máscara de filas cuyos grupos fueron seleccionados.

**Línea 4:** `.loc[...]` y `.copy()` crean entrenamiento.

**Línea 5:** `~` invierte la máscara y crea prueba. Ninguna fila cambia de grupo por su popularidad.

In [4]:
claves_unicas = base_unidad2["clave_cancion"].drop_duplicates().sort_values().reset_index(drop=True)
claves_entrenamiento = claves_unicas.sample(frac=0.80, random_state=42)
es_entrenamiento = base_unidad2["clave_cancion"].isin(claves_entrenamiento)
datos_entrenamiento = base_unidad2.loc[es_entrenamiento].copy()
datos_prueba = base_unidad2.loc[~es_entrenamiento].copy()

### 5.1 Contar las observaciones de cada conjunto

**Línea 1:** `pd.DataFrame({...}, index=[...])` reúne cantidad de IDs y claves por conjunto. `len(...)` cuenta filas; `.nunique()` cuenta grupos.

**Línea 2:** divide cada cantidad de IDs por el total y multiplica por 100.

**Línea 3:** redondea solo el resumen mostrado.

Aquí revisamos tamaño y pertenencia de la partición, **no medias, errores o rendimiento de popularidad de prueba**.

In [5]:
resumen_particion = pd.DataFrame({"IDs": [len(datos_entrenamiento), len(datos_prueba)], "Claves distintas": [datos_entrenamiento["clave_cancion"].nunique(), datos_prueba["clave_cancion"].nunique()]}, index=["Entrenamiento", "Prueba interna"])
resumen_particion["Porcentaje de IDs"] = resumen_particion["IDs"] / len(base_unidad2) * 100
resumen_particion.round(3)

,IDs,Claves distintas,Porcentaje de IDs
Entrenamiento,71728,64965,79.929
Prueba interna,18012,16241,20.071


**Resultado de la partición:** entrenamiento contiene **71.728 IDs (79,93%)** y prueba interna **18.012 IDs (20,07%)**. Los porcentajes no son exactos porque agrupamos claves con varios IDs.

La semilla reproduce la selección con esta base y el entorno usado. Si cambian la base o las versiones de herramientas, debe conservarse el manifiesto de partición para recuperar exactamente las mismas asignaciones.

### 5.2 Comprobar intersecciones y cobertura

**Línea 1:** `.index.intersection(...)` busca IDs compartidos.

**Línea 2:** `set(...).intersection(...)` compara conjuntos de claves sin importar su orden.

**Línea 3:** `.index.union(...)` reúne los IDs asignados a alguno de los conjuntos.

**Línea 4:** la Serie resume los dos solapamientos, el porcentaje de cobertura, la ausencia de IDs duplicados y la igualdad entre IDs de origen y asignados.

**Línea 5:** muestra los resultados. Un solo solapamiento implicaría revisar el procedimiento antes de modelar.

In [6]:
ids_compartidos = datos_entrenamiento.index.intersection(datos_prueba.index)
claves_compartidas = set(datos_entrenamiento["clave_cancion"]).intersection(set(datos_prueba["clave_cancion"]))
ids_asignados = datos_entrenamiento.index.union(datos_prueba.index)
comprobaciones_particion = pd.Series({
    "IDs compartidos": len(ids_compartidos),
    "Claves compartidas": len(claves_compartidas),
    "Cobertura de IDs (%)": len(ids_asignados) / len(base_unidad2) * 100,
    "IDs únicos en cada conjunto": datos_entrenamiento.index.is_unique and datos_prueba.index.is_unique,
    "Todos los IDs asignados sin adiciones": set(ids_asignados) == set(base_unidad2.index),
    "Suma de filas coincide con base": len(datos_entrenamiento) + len(datos_prueba) == len(base_unidad2)
})
comprobaciones_particion

IDs compartidos                              0
Claves compartidas                           0
Cobertura de IDs (%)                     100.0
IDs únicos en cada conjunto               True
Todos los IDs asignados sin adiciones     True
Suma de filas coincide con base           True
dtype: object

**Resultados esperados y comprobados:** cero IDs compartidos, cero claves compartidas y cobertura del 100%. Los controles de integridad son verdaderos.

Esto respalda que la separación respeta nuestra clave de agrupación. No prueba ausencia de toda fuga posible ni resuelve la exposición exploratoria previa del dataset.

## 6. Separar entradas X y objetivo y

**X:** características disponibles para estimar. **y:** popularidad que queremos aproximar.

**Línea 1:** enumera columnas que no se usarán como entradas: objetivo, clave auxiliar y duración en milisegundos redundante con minutos.

**Líneas 2 y 3:** `.drop(columns=...).copy()` construye X para ambos conjuntos sin tocar los datos originales.

**Líneas 4 y 5:** seleccionan popularidad como y, manteniendo los índices para alinear entradas y respuesta.

No se convierte artista o título en un número de popularidad. Los géneros mantienen Multi-Hot y el modelo no recibe la clave usada para separar.

In [7]:
columnas_excluir_X = ["popularity", "clave_cancion", "duration_ms"]
X_train = datos_entrenamiento.drop(columns=columnas_excluir_X).copy()
X_test = datos_prueba.drop(columns=columnas_excluir_X).copy()
y_train = datos_entrenamiento["popularity"].copy()
y_test = datos_prueba["popularity"].copy()

### 6.1 Verificar columnas y alineación

La Serie comprueba igual cantidad y orden de columnas de X, presencia de los 114 géneros y alineación de índices con y. También comprueba que objetivo, ID, clave y duración redundante estén fuera de las columnas.

`.columns.equals(...)` y `.index.equals(...)` comparan las etiquetas, no sus valores musicales. `startswith("genero__")` identifica el bloque Multi-Hot por prefijo. `all(...)` exige ausencia de los nombres prohibidos.

**Resultado:** ambos X tienen **127 columnas**: diez medidas, tres categorías y 114 géneros. Los IDs están en el índice para alinear, no en las columnas predictoras.

In [8]:
comprobaciones_X_y = pd.Series({
    "Columnas de X idénticas y en el mismo orden": X_train.columns.equals(X_test.columns),
    "127 columnas de entrada": X_train.shape[1] == 127 and X_test.shape[1] == 127,
    "114 indicadores de género": X_train.columns.str.startswith("genero__").sum() == 114,
    "Índices train alineados con y": X_train.index.equals(y_train.index),
    "Índices test alineados con y": X_test.index.equals(y_test.index),
    "Objetivo y auxiliares fuera de X": all(columna not in X_train.columns for columna in ["popularity", "track_id", "clave_cancion", "duration_ms"])
})
comprobaciones_X_y

Columnas de X idénticas y en el mismo orden    True
127 columnas de entrada                        True
114 indicadores de género                      True
Índices train alineados con y                  True
Índices test alineados con y                   True
Objetivo y auxiliares fuera de X               True
dtype: bool

## 7. Guardar la asignación para poder reproducirla

Guardaremos un archivo **nuevo**, `Spotify_Particion_Unidad_2.csv`, con ID, clave y conjunto. No contiene predicciones ni reemplaza el CSV preparado.

**Línea 1:** selecciona la clave en una copia; el ID sigue en el índice.

**Línea 2:** etiqueta inicialmente como prueba interna.

**Línea 3:** `.loc[es_entrenamiento, "conjunto"]` cambia a entrenamiento las filas de la máscara.

**Línea 4:** `.to_csv(..., index_label="track_id")` escribe el índice con ese encabezado. El nombre distinto evita sobrescribir la base de entrada.

**Línea 5:** cuenta asignaciones para verificar el guardado previsto. Este archivo ayuda a mantener una partición fija; no convierte la prueba interna en externa.

In [9]:
particion_unidad2 = base_unidad2[["clave_cancion"]].copy()
particion_unidad2["conjunto"] = "prueba_interna"
particion_unidad2.loc[es_entrenamiento, "conjunto"] = "entrenamiento"
particion_unidad2.to_csv("Spotify_Particion_Unidad_2.csv", index_label="track_id")
particion_unidad2["conjunto"].value_counts()

conjunto
entrenamiento     71728
prueba_interna    18012
Name: count, dtype: int64

## 8. Estado de los KPI y siguiente paso

| KPI | Resultado de este paso | Estado |
|---|---|---|
| IDs compartidos | 0 | Cumple |
| Claves artista + título compartidas | 0 | Cumple bajo la normalización descrita |
| Cobertura | 100% de los IDs preparados | Cumple |
| Mejora relativa de MAE ≥10% | No calculada | Pendiente de referencia, modelos y evaluación |

Los cumplimientos actuales son de integridad de la separación. **No hay todavía evidencia de desempeño predictivo.** No calculamos la mediana de referencia con toda la base ni transformamos medidas usando prueba.

**Antes de entrenar:** habrá que definir una validación agrupada dentro de entrenamiento, el tratamiento de ausencias y códigos pendientes y qué modelos necesitan codificación adicional o escalamiento. Las decisiones se escogerán con validación de entrenamiento; prueba se conserva para el informe final.

**Preguntas de comprensión:**

1. ¿Por qué el puntaje 0–100 plantea regresión y no obliga a crear clases?
2. ¿Por qué elegimos una referencia de mediana para comparar MAE?
3. ¿Por qué el 10% es una meta de trabajo y no una garantía comercial?
4. ¿Por qué separar 80% de grupos no garantiza exactamente 80% de IDs?
5. ¿Qué versiones reconoce nuestra clave y cuáles podría no reconocer?
6. ¿Por qué esta prueba es interna y no completamente ciega?
7. ¿Por qué no utilizamos las tablas de escalamiento demo de la unidad 1?

**Próximo paso:** preparar la validación dentro de entrenamiento y construir la referencia constante, explicando sus errores antes de incorporar el primer modelo.

## 9. Validación dentro del entrenamiento

**Objetivo de este paso:** obtener una referencia de error antes de entrenar modelos con características. Usaremos exclusivamente `datos_entrenamiento`, `X_train` y `y_train`.

La **prueba interna permanece reservada**. Su popularidad no se consulta para calcular medianas, construir pliegues, medir errores ni fijar el objetivo numérico de desarrollo.

**Validación cruzada de tres pliegues:** dividimos las claves de entrenamiento en tres partes. En cada ronda, dos partes permiten ajustar la referencia y la tercera permite evaluar. Repetimos para que cada ID de entrenamiento tenga exactamente una predicción realizada sin utilizar su respuesta para ajustar.

| Ronda | Ajuste de la mediana | Validación |
|---|---|---|
| 1 | Pliegues 2 y 3 | Pliegue 1 |
| 2 | Pliegues 1 y 3 | Pliegue 2 |
| 3 | Pliegues 1 y 2 | Pliegue 3 |

Los grupos de artista + título se mantienen completos en un solo pliegue. No es una separación por artistas completos ni una evaluación temporal.

### 9.1 Formar los pliegues por grupos

Se implementa la mecánica con Pandas para hacerla visible. Es una validación agrupada de tres pliegues, con asignación aleatoria reproducible por claves; no una llamada a `GroupKFold` ni una estratificación por popularidad.

**Línea 1:** fija tres pliegues. Son tres rondas, no tres modelos candidatos.

**Línea 2:** obtiene y ordena las claves distintas solo de entrenamiento; `.sample(frac=1, random_state=42)` mezcla todas, sin reemplazo; `reset_index` crea posiciones consecutivas.

**Línea 3:** `[posicion % numero_pliegues for posicion in range(...)]` reparte esas posiciones entre 0, 1 y 2. `%` devuelve el resto de la división; el patrón es 0,1,2,0,1,2… La Serie usa la clave como índice.

**Línea 4:** `.map(pliegue_por_clave)` asigna a cada ID el pliegue de su clave.

**Línea 5:** muestra cuántos IDs quedaron en cada pliegue.

Las claves se distribuyen casi por igual, pero los tamaños por ID pueden diferir porque una clave puede reunir varias versiones. No se utilizó popularidad para equilibrar los pliegues.

In [10]:
numero_pliegues = 3
claves_validacion = datos_entrenamiento["clave_cancion"].drop_duplicates().sort_values().sample(frac=1, random_state=42).reset_index(drop=True)
pliegue_por_clave = pd.Series([posicion % numero_pliegues for posicion in range(len(claves_validacion))], index=claves_validacion)
pliegue_por_id = datos_entrenamiento["clave_cancion"].map(pliegue_por_clave)
pliegue_por_id.value_counts().sort_index()

clave_cancion
0    23913
1    24104
2    23711
Name: count, dtype: int64

**Interpretación:** los códigos 0, 1 y 2 identifican los pliegues; al presentar rondas usaremos 1, 2 y 3. La suma de sus tamaños debe ser **71.728 IDs**, todo el entrenamiento.

El reparto contiene **23.913**, **24.104** y **23.711 IDs** respectivamente. Diferencias de tamaño pequeñas no son pérdida de datos.

### 9.2 Verificar pertenencia y grupos

**Línea 1:** se crea una tabla auxiliar con clave y pliegue por ID, alineados por índice.

**Línea 2:** `groupby("clave_cancion")["pliegue"].nunique()` cuenta cuántos pliegues distintos tiene cada clave. Debe ser uno.

**Línea 3:** la Serie comprueba asignación sin faltantes, solo códigos 0–2, una asignación por clave, tamaño total e índice alineado con `y_train`.

**Línea 4:** muestra los controles. Si alguno falla, detenemos el cálculo con la siguiente celda, en lugar de continuar con una referencia inválida.

In [11]:
asignacion_validacion = pd.DataFrame({"clave_cancion": datos_entrenamiento["clave_cancion"], "pliegue": pliegue_por_id})
pliegues_por_grupo = asignacion_validacion.groupby("clave_cancion")["pliegue"].nunique()
controles_validacion = pd.Series({
    "Todos los IDs de entrenamiento tienen pliegue": pliegue_por_id.notna().all(),
    "Solo tres códigos de pliegue": pliegue_por_id.isin([0, 1, 2]).all(),
    "Cada clave pertenece a un único pliegue": pliegues_por_grupo.eq(1).all(),
    "Asignaciones cubren el entrenamiento": len(asignacion_validacion) == len(datos_entrenamiento),
    "Índices alineados con y_train": pliegue_por_id.index.equals(y_train.index)
})
controles_validacion

Todos los IDs de entrenamiento tienen pliegue    True
Solo tres códigos de pliegue                     True
Cada clave pertenece a un único pliegue          True
Asignaciones cubren el entrenamiento             True
Índices alineados con y_train                    True
dtype: bool

In [12]:
if not controles_validacion.all():
    raise ValueError("Revisar la asignación de pliegues antes de evaluar la referencia.")

## 10. MAE explicado con un ejemplo pequeño

Antes del cálculo real, usamos **datos ficticios**. Un conjunto de ajuste con valores 10, 40 y 70 tiene mediana 40. La referencia predice 40 para todas las observaciones de validación, sin mirar sus respuestas para elegir esa constante.

| Popularidad real de validación | Predicción constante | Error absoluto |
|---|---|---|
| 20 | 40 | 20 |
| 50 | 40 | 10 |
| 80 | 40 | 40 |

**MAE = (20 + 10 + 40) / 3 = 23,33 puntos.**

Primero se resta y luego se toma valor absoluto. Promediar errores con signo permitiría que sobreestimaciones y subestimaciones se cancelaran, ocultando equivocaciones.

### 10.1 Calcular el ejemplo con Pandas

**Línea 1:** crea los tres valores ficticios para ajustar.

**Línea 2:** crea las respuestas ficticias de validación; son un objeto separado.

**Línea 3:** `.median()` aprende la constante solo del ejemplo de ajuste.

**Línea 4:** `pd.Series(constante, index=...)` repite esa predicción en el índice de validación, para alinear cada respuesta y predicción.

**Línea 5:** resta y `.abs()` convierte a error absoluto.

**Línea 6:** reúne reales, predicciones y errores en una tabla para leerlos juntos.

In [13]:
y_ajuste_ejemplo = pd.Series([10, 40, 70])
y_validacion_ejemplo = pd.Series([20, 50, 80])
mediana_ejemplo = y_ajuste_ejemplo.median()
predicciones_ejemplo = pd.Series(mediana_ejemplo, index=y_validacion_ejemplo.index)
errores_ejemplo = (y_validacion_ejemplo - predicciones_ejemplo).abs()
pd.DataFrame({"real": y_validacion_ejemplo, "prediccion": predicciones_ejemplo, "error_absoluto": errores_ejemplo})

,real,prediccion,error_absoluto
0,20,40.0,20.0
1,50,40.0,10.0
2,80,40.0,40.0


**Línea:** `errores_ejemplo.mean()` suma los tres errores absolutos y divide por la cantidad de observaciones. La salida puede mostrar muchos decimales: el resultado se interpreta como **23,33 puntos**, no como porcentaje de error relativo ni como porcentaje de aciertos.

In [14]:
errores_ejemplo.mean()

np.float64(23.333333333333332)

## 11. Referencia de mediana en cada ronda

La referencia no usa energía, géneros ni otras columnas de X. Todos los IDs de validación de una ronda reciben **la misma predicción**, la mediana aprendida con los otros dos pliegues.

¿Por qué mediana? Minimiza la suma de errores absolutos entre predictores constantes en los datos usados para ajustarla. Eso la hace coherente con MAE; no garantiza el menor error en datos nuevos.

No calculamos una única mediana con los 71.728 IDs para validar esas mismas observaciones. Cada ronda aprende su constante excluyendo las respuestas del pliegue que evalúa.

**Predicciones fuera de pliegue (OOF):** guardaremos una predicción por ID, realizada cuando ese ID y su clave estaban fuera del ajuste.

### 11.1 Preparar las estructuras de resultados

**Línea 1:** crea una Serie vacía con índice de `y_train` y tipo decimal. Sus posiciones se completarán con las predicciones fuera de pliegue. Se mantiene el orden de IDs.

**Línea 2:** crea una lista para los resultados por ronda. No se calcula aún ninguna mediana ni se utiliza prueba.

In [15]:
predicciones_mediana_oof = pd.Series(index=y_train.index, dtype="float64", name="prediccion_mediana_oof")
resultados_mediana = []

### 11.2 Ejecutar las rondas: primera parte de las instrucciones

El siguiente bloque repite el mismo procedimiento tres veces:

1. `for pliegue in range(numero_pliegues):` recorre códigos 0, 1 y 2.
2. `.eq(pliegue)` marca los IDs que validaremos en esa ronda.
3. `y_train.loc[~es_validacion]` selecciona respuestas para ajuste; `~` invierte la máscara.
4. `y_train.loc[es_validacion]` selecciona respuestas de validación.
5. Las dos llamadas `set(...)` obtienen las claves de cada parte.
6. `if claves_ajuste.intersection(claves_evaluadas):` comprueba que no haya claves compartidas; un conjunto vacío equivale a falso.
7. `raise ValueError(...)` detendría la ejecución si se detectara un solapamiento.

La validación interna conserva la misma restricción de grupos que la separación inicial.

### Segunda parte de las instrucciones

8. `y_ajuste.median()` calcula la única constante de la ronda, sin respuestas de validación.
9. `pd.Series(..., index=y_validacion.index)` construye predicciones alineadas para esa validación.
10. `(y_validacion - prediccion_pliegue).abs()` calcula un error absoluto por ID.
11. `.mean()` calcula el MAE de la ronda.
12. `.loc[y_validacion.index] = ...` guarda esas predicciones en su lugar de la Serie OOF.
13. `.append({...})` registra ronda, tamaños, mediana, MAE y conteo de claves compartidas. `pliegue + 1` hace la etiqueta más legible, sin cambiar la división.

Las respuestas de validación solo se utilizan en el paso de medir errores; no entran al ajuste de su mediana.

In [16]:
for pliegue in range(numero_pliegues):
    es_validacion = pliegue_por_id.eq(pliegue)
    y_ajuste = y_train.loc[~es_validacion]
    y_validacion = y_train.loc[es_validacion]
    claves_ajuste = set(datos_entrenamiento.loc[~es_validacion, "clave_cancion"])
    claves_evaluadas = set(datos_entrenamiento.loc[es_validacion, "clave_cancion"])
    if claves_ajuste.intersection(claves_evaluadas):
        raise ValueError("Hay claves compartidas entre ajuste y validación.")
    mediana_ajuste = y_ajuste.median()
    prediccion_pliegue = pd.Series(mediana_ajuste, index=y_validacion.index)
    errores_pliegue = (y_validacion - prediccion_pliegue).abs()
    mae_pliegue = errores_pliegue.mean()
    predicciones_mediana_oof.loc[y_validacion.index] = prediccion_pliegue
    resultados_mediana.append({"Ronda": pliegue + 1, "IDs ajuste": len(y_ajuste), "IDs validación": len(y_validacion), "Mediana de ajuste": mediana_ajuste, "MAE validación": mae_pliegue, "Claves compartidas": len(claves_ajuste.intersection(claves_evaluadas))})

### 11.3 Mostrar los resultados reales

**Línea 1:** `pd.DataFrame(resultados_mediana)` transforma los diccionarios en una tabla, una fila por ronda.

**Línea 2:** `.round(4)` redondea la presentación a cuatro decimales, sin reemplazar los cálculos originales.

La columna «Mediana de ajuste» es una predicción constante; «MAE validación» es el error de esa constante en observaciones que no la ajustaron. No son el mismo número.

In [17]:
tabla_referencia_mediana = pd.DataFrame(resultados_mediana)
tabla_referencia_mediana.round(4)

,Ronda,IDs ajuste,IDs validación,Mediana de ajuste,MAE validación,Claves compartidas
0,1,47815,23913,33.0,17.3627,0
1,2,47624,24104,34.0,17.3657,0
2,3,48017,23711,33.0,17.0243,0


### Cómo interpretar las tres rondas

Las medianas aprendidas fueron **33,00, 34,00, 33,00 puntos**. Los MAE fueron **17,363; 17,366; 17,024 puntos**, respectivamente.

En cada ronda, tamaños de ajuste y validación suman 71.728 IDs. Las partes cambian entre rondas y cada ID valida una sola vez. Las claves compartidas son cero.

Un MAE cercano a 17,252 significa equivocarse en promedio unos 17,252 puntos de popularidad al predecir una constante, según esta validación. No significa que todas las predicciones estén dentro de esa distancia.

Este resultado es una **referencia interna**, no el error del conjunto de prueba ni una estimación de éxito futuro garantizada.

## 12. Error fuera de pliegue para todo el entrenamiento

**Línea 1:** `pd.Series({...})` comprueba que cada posición OOF tenga una predicción, que el índice siga alineado, que las validaciones cubran todos los IDs y que no existan claves compartidas por ronda.

**Línea 2:** muestra los controles. Son comprobaciones de procedimiento, no porcentajes de acierto.

Si alguna predicción faltara, no sería correcto dejar que `.mean()` la ignorara silenciosamente y presentara el error como si cubriera todo el entrenamiento.

In [18]:
controles_oof = pd.Series({
    "Todos los IDs tienen predicción OOF": predicciones_mediana_oof.notna().all(),
    "Índice OOF alineado con y_train": predicciones_mediana_oof.index.equals(y_train.index),
    "Validaciones cubren todo el entrenamiento": tabla_referencia_mediana["IDs validación"].sum() == len(y_train),
    "Ninguna ronda comparte claves": tabla_referencia_mediana["Claves compartidas"].eq(0).all()
})
controles_oof

Todos los IDs tienen predicción OOF          True
Índice OOF alineado con y_train              True
Validaciones cubren todo el entrenamiento    True
Ninguna ronda comparte claves                True
dtype: bool

### 12.1 Calcular el MAE global OOF

**Líneas 1–2:** detienen el cálculo si falla algún control.

**Línea 3:** obtiene el error absoluto de cada ID con su predicción fuera de pliegue.

**Línea 4:** promedia los errores de todos los IDs, dando a cada uno el mismo peso.

**Línea 5:** muestra el número sin confundirlo con el tamaño de la muestra.

Este será el valor principal de referencia para comparar modelos que utilicen los mismos pliegues.

In [19]:
if not controles_oof.all():
    raise ValueError("Las predicciones OOF no cubren correctamente el entrenamiento.")
errores_mediana_oof = (y_train - predicciones_mediana_oof).abs()
mae_referencia_oof = errores_mediana_oof.mean()
mae_referencia_oof

np.float64(17.251826343966094)

**Resultado real: MAE OOF = 17,252 puntos**, calculado sobre las 71.728 predicciones de entrenamiento fuera de pliegue.

Todavía no podemos calificarlo como suficiente para un uso concreto: falta una tolerancia de error de negocio. Sí podemos exigir que un modelo con características aporte una mejora respecto de esta referencia.

### 12.2 Promedio de rondas, dispersión y promedio ponderado

**Línea 1:** promedia los tres MAE, dando a cada ronda igual peso.

**Línea 2:** `.std(ddof=0)` calcula la desviación de esos tres resultados con divisor tres; describe variación entre estas rondas.

**Línea 3:** pondera cada MAE por la cantidad de IDs que validó y divide por todos los IDs validados. Debe coincidir con el MAE OOF salvo precisión numérica.

**Línea 4:** muestra los tres resúmenes y el MAE OOF.

Los tamaños no son exactamente iguales, por eso la media simple puede diferir ligeramente del error por ID. Ninguno es un intervalo de confianza ni el margen de error de una canción.

In [20]:
mae_medio_rondas = tabla_referencia_mediana["MAE validación"].mean()
desviacion_mae_rondas = tabla_referencia_mediana["MAE validación"].std(ddof=0)
mae_ponderado_rondas = (tabla_referencia_mediana["MAE validación"] * tabla_referencia_mediana["IDs validación"]).sum() / tabla_referencia_mediana["IDs validación"].sum()
pd.Series({"Media de MAE por ronda": mae_medio_rondas, "Desviación entre rondas (ddof=0)": desviacion_mae_rondas, "MAE ponderado por IDs": mae_ponderado_rondas, "MAE OOF por ID": mae_referencia_oof})

Media de MAE por ronda              17.250883
Desviación entre rondas (ddof=0)     0.160243
MAE ponderado por IDs               17.251826
MAE OOF por ID                      17.251826
dtype: float64

**Resultados:** media de rondas **17,251**, desviación entre rondas **0,160** y MAE ponderado **17,252**.

La variación entre estas rondas es pequeña respecto del tamaño del error medio. No basta para demostrar ausencia de sesgos, fugas o sobreajuste; una constante puede variar poco y seguir cometiendo errores importantes.

## 13. Relacionar la referencia con el KPI de mejora

La meta definida es mejorar al menos un 10% frente a la referencia. Para orientar el desarrollo con los mismos pliegues, eso equivale a un MAE del futuro modelo igual o inferior al 90% del MAE OOF de referencia.

**Línea 1:** fija la meta relativa como fracción, 0,10.

**Línea 2:** multiplica la referencia por `1 - 0.10` para obtener el umbral de desarrollo.

**Línea 3:** muestra referencia, porcentaje propuesto y umbral resultante.

No se cambió la meta después de observar los resultados. Solo se traduce el 10% ya fijado a puntos usando una referencia calculada dentro de entrenamiento.

In [21]:
mejora_objetivo = 0.10
mae_objetivo_validacion = mae_referencia_oof * (1 - mejora_objetivo)
pd.Series({"MAE de referencia OOF": mae_referencia_oof, "Mejora objetivo (%)": mejora_objetivo * 100, "MAE máximo del modelo para esa meta en validación": mae_objetivo_validacion})

MAE de referencia OOF                                17.251826
Mejora objetivo (%)                                  10.000000
MAE máximo del modelo para esa meta en validación    15.526644
dtype: float64

**Umbral orientativo de desarrollo: 15,527 puntos de MAE OOF o menos**, usando las mismas observaciones y los mismos pliegues.

**El KPI sigue pendiente:** no existe todavía un modelo con características que comparar. La referencia comparada consigo misma tendría mejora cero, no 10%.

Al realizar la evaluación final, se compararán referencia y modelo sobre los mismos datos de prueba y con sus propias métricas de prueba. No se trasladará el MAE OOF como si fuera el error de referencia en prueba. Ahora no calculamos esa evaluación.

## 14. Guardar los pliegues de entrenamiento

**Línea 1:** crea un manifiesto nuevo a partir de la tabla de asignación de validación.

**Línea 2:** transforma los códigos internos 0–2 en etiquetas 1–3 para guardarlos de forma legible.

**Línea 3:** exporta `Spotify_Validacion_Entrenamiento.csv`, con ID como campo. Solo contiene IDs de entrenamiento, clave y pliegue; no respuestas, predicciones ni registros de prueba.

**Línea 4:** muestra los conteos del manifiesto.

Conservarlo permite que los futuros modelos utilicen las mismas rondas, sin seleccionar otra semilla para buscar mejores resultados.

In [22]:
manifiesto_validacion = asignacion_validacion.copy()
manifiesto_validacion["pliegue"] = manifiesto_validacion["pliegue"] + 1
manifiesto_validacion.to_csv("Spotify_Validacion_Entrenamiento.csv", index_label="track_id")
manifiesto_validacion["pliegue"].value_counts().sort_index()

pliegue
1    23913
2    24104
3    23711
Name: count, dtype: int64

## 15. Cierre de este paso

| Elemento | Resultado |
|---|---|
| Validación | Tres pliegues agrupados por artista + título dentro de entrenamiento |
| Cobertura | 71.728 IDs; una predicción OOF por ID |
| Solapamiento de claves entre ajuste y validación | Cero en las tres rondas |
| Referencia | Mediana aprendida solo con los otros dos pliegues |
| MAE OOF de referencia | 17,252 puntos |
| Meta de desarrollo para mejorar 10% | MAE OOF ≤ 15,527 puntos |
| Conjunto de prueba | Reservado; sin errores ni resúmenes de popularidad calculados en este paso |

**Qué aprendimos:** MAE es el promedio de tamaños de error; la referencia no aprende relaciones con características; y las predicciones OOF permiten medirla sin usar cada respuesta para ajustar su propia constante.

**Limitaciones:** la validación no es externa ni temporal. Se mantienen la exposición exploratoria previa de la unidad 1, las limitaciones de la clave y las discrepancias de popularidad resumidas. No se entrenó todavía una regresión con X ni se ajustó la imputación o el escalamiento.

**Preguntas de comprensión:**

1. ¿Qué diferencia hay entre entrenamiento, ajuste de una ronda, validación y prueba?
2. ¿Por qué calculamos tres medianas en lugar de una sobre todo `y_train` para evaluar sus propias observaciones?
3. ¿Por qué agrupamos claves y no distribuimos sus versiones libremente?
4. ¿Qué significa una predicción fuera de pliegue?
5. ¿Por qué el MAE no permite cancelar errores negativos y positivos?
6. ¿Por qué la desviación de MAE entre rondas no es un margen de error por canción?
7. ¿Por qué el KPI de mejora sigue pendiente?

**Próximo paso:** preparar dentro de cada ajuste las variables que necesita el primer modelo, entrenarlo con los mismos pliegues y comparar sus predicciones con esta referencia. Prueba seguirá reservada hasta seleccionar el procedimiento.